# Pre-training với PhysioNet BIDMC Respiration Dataset

Notebook này tải xuống và train model trên **PhysioNet BIDMC PPG and Respiration Dataset**
để khởi tạo trọng số model trước khi fine-tune bằng dữ liệu CSI thực tế của mình.

## Tại sao dùng PhysioNet?
- Tín hiệu `RESP` trong BIDMC là **dạng sóng nhịp thở thực tế** (tương đương output của `processor.py`)
- 53 bệnh nhân × ~8 phút = ~400 phút dữ liệu → dataset lớn để pre-train
- **Không cần sửa code hiện tại**: dữ liệu được cắt thành windows 30 giây giống hệt `train_model.ipynb`

## Flow
```
PhysioNet BIDMC (RESP signal)
  ↓ Resample → 100Hz (khớp với SAMPLING_RATE ESP32)
  ↓ Sliding Windows 30s
  ↓ Tính LabelBPM từ annotations
  ↓ Train 1D-CNN + LSTM (cùng kiến trúc train_model.ipynb)
  ↓ Lưu models/pretrained_physionet.h5
  ↓ Fine-tune bằng collect_data.py của bạn → models/respiration_model.h5
```

## Output
- `models/pretrained_physionet.h5`  : Model đã pre-train (dùng để fine-tune)
- `models/pretrain_report.png`      : Báo cáo đánh giá (4 Panel)
- `asset/pretrain_report.png`       : Copy sang asset/

## 0. Cài đặt

In [ ]:
!pip install wfdb tensorflow scikit-learn numpy scipy matplotlib -q

## 1. Import & Config

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import resample, butter, filtfilt, find_peaks
from sklearn.model_selection import train_test_split
import wfdb

MODELS_DIR    = '../models'
ASSET_DIR     = '../asset'
SAMPLING_RATE = 100.0      # Hz — phải khớp với firmware ESP32
WINDOW_SEC    = 30
WINDOW_SIZE   = int(SAMPLING_RATE * WINDOW_SEC)  # 3000
STEP_SEC      = 5
STEP_SIZE     = int(SAMPLING_RATE * STEP_SEC)    # 500
BPM_MIN, BPM_MAX = 4.0, 40.0

# PhysioNet BIDMC: 53 bản ghi (bidmc01 → bidmc53)
BIDMC_RECORDS = [f'bidmc{i:02d}' for i in range(1, 54)]
DB_NAME = 'bidmc'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ASSET_DIR, exist_ok=True)
print(f'Config: WINDOW={WINDOW_SEC}s @ {SAMPLING_RATE}Hz = {WINDOW_SIZE} samples')
print(f'Số bản ghi BIDMC: {len(BIDMC_RECORDS)}')

## 2. Download & Xử lý PhysioNet BIDMC

BIDMC dataset có tín hiệu `RESP` được lấy mẫu ở 125Hz, ta resample xuống 100Hz.
BPM Ground Truth được tính từ khoảng cách giữa các đỉnh nhịp thở.

In [ ]:
def bpm_from_peaks(signal: np.ndarray, fs: float) -> float:
    """
    Tính BPM bằng cách đếm đỉnh trong tín hiệu nhịp thở.
    Logic giống Peak Detection trong estimator.py.
    """
    duration_sec = len(signal) / fs
    # Khoảng cách tối thiểu giữa 2 đỉnh = 1.5s (tương đương 40 BPM max)
    min_distance = int(fs * 1.5)
    peaks, _ = find_peaks(signal, distance=min_distance, height=np.std(signal) * 0.3)
    n_peaks = len(peaks)
    if n_peaks < 2:
        return 0.0
    bpm = (n_peaks / duration_sec) * 60.0
    return round(bpm, 2)


def bandpass_resp(signal: np.ndarray, fs: float,
                  low: float = 0.1, high: float = 0.8) -> np.ndarray:
    """Band-pass tín hiệu RESP (dải rộng hơn vì BIDMC có bệnh nhân thở nhanh)."""
    nyq = fs / 2.0
    b, a = butter(4, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, signal)


def load_bidmc_record(record_name: str):
    """
    Tải 1 bản ghi BIDMC từ PhysioNet.
    Trả về (resp_100hz, fs_original) hoặc None nếu lỗi.
    """
    try:
        # Đọc trực tiếp từ PhysioNet (không cần download thủ công)
        record = wfdb.rdrecord(record_name, pn_dir=DB_NAME)
    except Exception as e:
        print(f'[SKIP] {record_name}: {e}')
        return None, None

    fs_orig = record.fs  # Thường là 125Hz

    # Tìm kênh RESP
    sig_names = [s.lower() for s in record.sig_name]
    resp_idx = None
    for candidate in ['resp', 'respiration', 'br']:
        if candidate in sig_names:
            resp_idx = sig_names.index(candidate)
            break

    if resp_idx is None:
        print(f'[SKIP] {record_name}: Không tìm thấy kênh RESP. Kênh có: {record.sig_name}')
        return None, None

    resp_signal = record.p_signal[:, resp_idx]
    resp_signal = resp_signal[~np.isnan(resp_signal)]  # Loại NaN

    # Resample từ fs_orig → 100Hz
    target_len = int(len(resp_signal) * SAMPLING_RATE / fs_orig)
    resp_100hz = resample(resp_signal, target_len)

    return resp_100hz, fs_orig


print('Test tải 1 bản ghi đầu tiên...')
test_sig, test_fs = load_bidmc_record('bidmc01')
if test_sig is not None:
    print(f'[OK] bidmc01: {len(test_sig)/SAMPLING_RATE:.0f}s @ {test_fs}Hz → resample → {SAMPLING_RATE}Hz')
    print(f'     Chiều dài: {len(test_sig)} samples')

In [ ]:
# ── Tải và xử lý toàn bộ 53 bản ghi ─────────────────────
X_list, y_list = [], []
skipped = 0

for rec in BIDMC_RECORDS:
    resp, fs_orig = load_bidmc_record(rec)
    if resp is None:
        skipped += 1; continue

    # Band-pass tín hiệu (step này thay thế processor.py cho dữ liệu không phải CSI)
    resp_filtered = bandpass_resp(resp, SAMPLING_RATE)

    if len(resp_filtered) < WINDOW_SIZE:
        print(f'[SKIP] {rec}: Quá ngắn ({len(resp_filtered)/SAMPLING_RATE:.0f}s)')
        skipped += 1; continue

    # Sliding Window (giống train_model.ipynb)
    n_windows = 0
    for start in range(0, len(resp_filtered) - WINDOW_SIZE + 1, STEP_SIZE):
        window = resp_filtered[start: start + WINDOW_SIZE]
        bpm = bpm_from_peaks(window, SAMPLING_RATE)

        if bpm < BPM_MIN or bpm > BPM_MAX:
            continue  # Loại bỏ windows có BPM bất thường

        # Chuẩn hóa về [-1, 1] (khác với CSI nhưng giữ thông tin hình dạng sóng)
        window_norm = (window - window.mean()) / (window.std() + 1e-8)
        X_list.append(window_norm)
        y_list.append(bpm)
        n_windows += 1

    if n_windows > 0:
        print(f'[OK] {rec}: {n_windows} windows')

X = np.array(X_list)
y = np.array(y_list)
print(f'\n===== DATASET =====')
print(f'Tổng: {len(X)} windows | Bỏ qua: {skipped} records')
print(f'BPM: min={y.min():.1f}, max={y.max():.1f}, mean={y.mean():.2f}')

## 3. Visualize Dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('PhysioNet BIDMC Dataset Overview', fontsize=13, fontweight='bold')

axes[0].hist(y, bins=20, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(12, color='green', linestyle='--', alpha=0.7, label='Normal min')
axes[0].axvline(20, color='orange', linestyle='--', alpha=0.7, label='Normal max')
axes[0].set_title('BPM Distribution (PhysioNet BIDMC)')
axes[0].set_xlabel('BPM'); axes[0].set_ylabel('Count'); axes[0].legend()

t = np.arange(WINDOW_SIZE) / SAMPLING_RATE
axes[1].plot(t, X[0], color='#F44336', linewidth=0.8)
axes[1].set_title(f'Example Breathing Signal (BPM={y[0]:.1f})')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Amplitude')

axes[2].plot(t, X[len(X)//2], color='#4CAF50', linewidth=0.8)
axes[2].set_title(f'Example (BPM={y[len(y)//2]:.1f})')
axes[2].set_xlabel('Time (s)')

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/physionet_data_overview.png', dpi=150)
plt.show()

## 4. Train / Test Split

In [ ]:
X_cnn = X[..., np.newaxis]  # (n, WINDOW_SIZE, 1)
y_norm = (y - BPM_MIN) / (BPM_MAX - BPM_MIN)

X_train, X_test, y_train, y_test = train_test_split(
    X_cnn, y_norm, test_size=0.15, random_state=42, shuffle=True
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 5. Xây dựng Model (Cùng kiến trúc với train_model.ipynb)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.random.set_seed(42)

def build_model(input_length):
    """Cùng kiến trúc 1D-CNN + LSTM với train_model.ipynb."""
    inp = keras.Input(shape=(input_length, 1), name='breathing_signal')
    x = layers.Conv1D(32, 50, strides=5, activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 25, strides=2, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid', name='bpm')(x)
    return keras.Model(inp, out, name='respiration_cnn_lstm')

model = build_model(WINDOW_SIZE)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

## 6. Training

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint(
        f'{MODELS_DIR}/pretrained_physionet.h5', save_best_only=True, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    epochs=80, batch_size=32, validation_split=0.15,
    callbacks=callbacks, verbose=1
)

np.savez(f'{MODELS_DIR}/physionet_training_history.npz',
    loss=history.history['loss'],
    val_loss=history.history['val_loss'],
    mae=history.history['mae'],
    val_mae=history.history['val_mae']
)
print('[OK] Đã lưu history và model pre-trained.')

## 7. Đánh giá — Training Report (4 Panel)

In [ ]:
best = keras.models.load_model(f'{MODELS_DIR}/pretrained_physionet.h5')

y_pred_norm = best.predict(X_test).flatten()
y_pred_bpm  = y_pred_norm * (BPM_MAX - BPM_MIN) + BPM_MIN
y_true_bpm  = y_test * (BPM_MAX - BPM_MIN) + BPM_MIN
errors      = y_pred_bpm - y_true_bpm
mae_final   = np.mean(np.abs(errors))
rmse_final  = np.sqrt(np.mean(errors**2))
within_2    = np.mean(np.abs(errors) <= 2.0) * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pre-training Report — PhysioNet BIDMC', fontsize=14, fontweight='bold')

# A: Loss
axes[0,0].plot(history.history['loss'], '#2196F3', label='Train Loss')
axes[0,0].plot(history.history['val_loss'], '#F44336', label='Val Loss')
best_ep = np.argmin(history.history['val_loss'])
axes[0,0].axvline(best_ep, color='green', linestyle=':', label=f'Best={best_ep}')
axes[0,0].set_title('(A) Loss (MSE)'); axes[0,0].set_xlabel('Epoch')
axes[0,0].legend(); axes[0,0].grid(True, linestyle='--', alpha=0.4)

# B: MAE in BPM
axes[0,1].plot(np.array(history.history['mae'])*(BPM_MAX-BPM_MIN), '#4CAF50', label='Train MAE')
axes[0,1].plot(np.array(history.history['val_mae'])*(BPM_MAX-BPM_MIN), '#FF9800', label='Val MAE')
axes[0,1].axhline(2.0, color='red', linestyle='--', label='Target ≤ 2 BPM')
axes[0,1].set_title('(B) MAE (BPM)'); axes[0,1].set_xlabel('Epoch')
axes[0,1].legend(); axes[0,1].grid(True, linestyle='--', alpha=0.4)

# C: Predicted vs True
axes[1,0].scatter(y_true_bpm, y_pred_bpm, alpha=0.5, color='#2196F3', s=20, edgecolors='none')
lim = [0, max(y_true_bpm.max(), y_pred_bpm.max()) + 3]
axes[1,0].plot(lim, lim, 'k--', alpha=0.5)
axes[1,0].fill_between(lim, [l-2 for l in lim], [l+2 for l in lim], alpha=0.08, color='green', label='±2 BPM')
axes[1,0].set_title(f'(C) Predicted vs True\nMAE={mae_final:.2f} | RMSE={rmse_final:.2f}')
axes[1,0].set_xlabel('True BPM'); axes[1,0].set_ylabel('Pred BPM')
axes[1,0].legend(); axes[1,0].grid(True, linestyle='--', alpha=0.3)

# D: Error Distribution
axes[1,1].hist(errors, bins=30, color='#9C27B0', edgecolor='white', alpha=0.8)
axes[1,1].axvline(0, color='black', linestyle='--')
axes[1,1].axvline(-2, color='green', linestyle=':'); axes[1,1].axvline(+2, color='green', linestyle=':')
axes[1,1].set_title(f'(D) Error Distribution\n{within_2:.1f}% within ±2 BPM')
axes[1,1].set_xlabel('Error (BPM)'); axes[1,1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/pretrain_report.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{ASSET_DIR}/pretrain_report.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n===== KẾT QUẢ PRE-TRAINING =====')
print(f'MAE  : {mae_final:.2f} BPM')
print(f'RMSE : {rmse_final:.2f} BPM')
print(f'Trong ±2 BPM: {within_2:.1f}%')
print(f'→ Lưu tại: {MODELS_DIR}/pretrained_physionet.h5')

## 8. Fine-tune bằng dữ liệu CSI của bạn

Sau khi thu thập đủ data bằng `collect_data.py`, chạy đoạn code này để fine-tune model pre-trained:

> **Lưu ý**: Bước này thay thế phần **Train** trong `train_model.ipynb`.
> Chỉ cần load pretrained model rồi tiếp tục train với learning rate thấp hơn.

In [ ]:
# ── Hướng dẫn Fine-tune (chạy TRONG train_model.ipynb sau khi có CSI data) ──
#
# Thay thế cell build_model() trong train_model.ipynb bằng:
#
# model = keras.models.load_model('../models/pretrained_physionet.h5')
# model.compile(
#     optimizer=keras.optimizers.Adam(1e-4),  # LR thấp hơn 10x khi fine-tune
#     loss='mse', metrics=['mae']
# )
#
# Sau đó chạy model.fit() như bình thường với CSI dataset.
# Model đã học được "hình dạng sóng nhịp thở" từ PhysioNet → fine-tune nhanh hơn.

print('Để fine-tune, mở train_model.ipynb và làm theo hướng dẫn trong cell này.')
print(f'Model pre-trained: {MODELS_DIR}/pretrained_physionet.h5')

# Lưu metadata pre-training để tham khảo
pretrain_meta = {
    'dataset': 'PhysioNet BIDMC PPG and Respiration',
    'n_windows': int(len(X)),
    'window_sec': WINDOW_SEC,
    'sampling_rate': SAMPLING_RATE,
    'bpm_min': BPM_MIN, 'bpm_max': BPM_MAX,
    'pretrain_mae_bpm': round(float(mae_final), 2),
    'pretrain_rmse_bpm': round(float(rmse_final), 2),
    'within_2bpm_pct': round(float(within_2), 1)
}
with open(f'{MODELS_DIR}/pretrain_metadata.json', 'w') as f:
    json.dump(pretrain_meta, f, indent=2)
print('Metadata:', json.dumps(pretrain_meta, indent=2))